# Trajectories {#sec-trajectories}

This tool is particularly useful to visualize convergence and deterministic trajectories along one dimension. In particular, it is suited to environments with only one agent and/or one state, such as a two-armed bandit task, that we use below as an example. 

First, we import what we need:

In [ ]:
 #| default_exp Utils.Trajectories1D

In [ ]:
 #| hide
# Imports for the nbdev development environment
import nbdev
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
 #| export
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import itertools as it

from collections.abc import Callable

from pyDOE import lhs

from fastcore.utils import *

In [ ]:
# imports for testing
from pyDDRL.Environments.TwoArmedBandit import TwoArmedBandit
from pyCRLD.Agents.Base import abase
from pyCRLD.Agents.StrategySARSA import stratSARSA

## Implementation

In [ ]:
 #| export
def trajectories_1D(mae,  # CRLD multi-agent environment object
                    x:tuple,  # which phase space axes to plot along x axes
                    res,  # specify range & resolution
                    NrRandom:int=3,  # how many random (in the other dimensions) stratgies for averaging 
                    traj_len:int=500,  # trajectory length
                    # col:str='LEN',  # color indicates either strength of flow via colormap, otherwise a fixed color name
                    cmap='viridis',  # Colormap
                    # kind='quiver+samples', # Kind of plot: "streamplot", "quiver+samples", "quiver", ...
                    # sf=0.5,  # Scale factor for quiver arrows
                    # lw=1.0,  # Line width for streamplot
                    # dens=0.75,  # Density for streamplot
                    acts=None,  # Action descriptions
                    conds=None,  # Conditions descriptions 
                    axes:Iterable=None,  # Axes to plot into
                    verbose=False,  # shall I talk to you while working?
                    ):  
    """
    Create a flow plot in strategy space in one dimension.
    """
    # Checks and balances
    # xlens, amx, lens = _checks_and_balances(x, y)
    
    # Fig and Axes
    if axes==None:
        states = mae.Z
        if mae.Z == 1:
            fig, ax = plt.subplots(1, states, figsize=(states * 3.2, 2.8))
            axes = [ax]
        else:
            fig, axes = plt.subplots(1, states, figsize=(states * 3.2, 2.8))
    else:
        axes = axes
        
    cmap = mpl.cm.get_cmap(cmap)
    ext = np.linspace(0, 1, len(res))
    colors = cmap(tuple(ext))
    
    # x axis
    xs = np.arange(0, traj_len)

    # The Plots
    for i in range(len(x[1])): # go through each plot sequentially 
        xinds = (x[0][0], x[1][i], x[2][0])  # fix this later
        # obtain the data to plot
        trajs = _data_to_plot(mae, res, xinds, traj_len, NrRandom, 
                              phasespace_items=_strategies, verbose=verbose)
        for j in range(len(trajs[i, :, 0])):
            # do the plot
            color = colors[j]
            axes[i].plot(xs, trajs[i, j, :], color=color)
        
    # Decorations
    # lens = max(xlens)
    if acts is None:
        acts = [f'act.{i}' for i in range(mae.M)]
        acts = mae.env.Aset
    if conds is None:
        conds = [f'state {i}' for i in range(mae.Z)]
    for i in range(len(x[1])):
        axes[i].set_title(conds[i])
        axes[i].set_xlabel("step")
    axes[0].set_ylabel("prob. act. "+str(xinds[2]))
    
    return axes

In [ ]:
 #| export
def _data_to_plot(mae,  # CRLD multi-agent environment object
                  res:Iterable,  # range & resolution of flow arrows 
                  xinds:tuple,  # of indices of the phase space object to plot along the x axis
                  traj_len:int,  # trajectory len
                  NrRandom:int,  # how many random (in the other dimensions) stratgies for averaging 
                  # difffunc:Callable,  # to compute which kind of arrows to plot (RPE or dX)
                  phasespace_items:Callable,  # to obtain phase space items for one ax plot point
                  verbose=False  # shall I talk to you while working?
                  ):
        
    l = len(res)
    trajs = np.zeros((mae.Z, l, traj_len))

    for i in range(mae.Z):
        for j, xval in enumerate(res):
            Xs = _strategies(mae, xinds, xval, NrRandom)  # generate random strategies to average over
            xtrajs = np.zeros((NrRandom, traj_len))
            for k, xk in enumerate(Xs):
                traj, fpreached = mae.trajectory(xk, traj_len) # compute trajectory
                xtrajs[k, :] = traj[:, xinds[0], i, xinds[2]]  # compute trajectory
            trajs[i, j, :] = np.mean(xtrajs, axis=0)  # add trajectory
            
    return trajs

In [ ]:
 #| export
def _strategies(mae,  # CRLD multi-agent environment object
                xinds:tuple,  # of indices of the phase space item to plot along the x axis
                xval:float,  # the value of the phase space item to plot along the x axis
                NrRandom,  # how many random (in the other dimensions) stratgies for averaging
               ) -> np.ndarray: # Array of joint strategies
    """
    Creates strategies (as a particular type of phase space item) for one ax plot point.
    All strategies have value `xval` at the `xinds` index and value `yval` at the `yinds`. 
    """   
    N, C, M = mae.N, mae.Q, mae.M  # Number of agents, conditions, actions
    # Xs = np.random.rand(NrRandom, N, C, M)  # random policies
    Xs = lhs(N*C*M, NrRandom).reshape(NrRandom, N, C, M)  # using latin hypercube sampling
    Xs = Xs / Xs.sum(axis=-1, keepdims=True)  # properly normalised
    
    xi, xc, xa = xinds; xa_ = tuple(set(range(M)) - set([xa]))
    
    Xs[:, xi, xc, xa] = xval  # setting x and y values
    
    # normalisation
    Xs[:,xi,xc,xa_] = (1-Xs[0, xi, xc, xa]) * Xs[:,xi,xc,xa_]\
        / np.sum(Xs[:,xi,xc,xa_], axis=-1, keepdims=True)
    
    return Xs

## Example with a two-armed bandit task

Let's consider a two-armed bandit task with $p_1 = 0.7$ and $p_2 = 0.3$. 

In [ ]:
# Define environment
xs = np.array([[-1, 1],
               [-1, 1]])
ps = np.array([[0.3, 0.7],
               [0.7, 0.3]])
tab = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)

Now, we define an agent-environment interface comprised of a SARSA agent playing the aforementioned task. This agent has a learning rate $\alpha=0.1$, a discount factor $\gamma = 0$, and a choice intensity $\beta = 4$. 

In [ ]:
# Agent-environment interface
aei = stratSARSA(tab,
                 learning_rates=0.1,
                 discount_factors=0,
                 choice_intensities=4.,
                 use_prefactor=False)

In [ ]:
x = ([0], [0], [0])  # (i,s,a) to plot along the x-axis
res = np.linspace(0, 1., 11)  # initial strategies
res[0] = 0.0001
res[-1] = 0.9999

Now plotting the trajectories:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
trajectories_1D(aei, x, res, NrRandom=1, traj_len=70, axes=[ax])
ax.set_title("deterministic trajectories")
ax.set_xlabel("time step "+r'$t$')
ax.set_ylabel("probability to choose arm 1")
plt.show()

/tmp/ipykernel_10416/1852574533.py:35: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = mpl.cm.get_cmap(cmap)


<Figure size 1200x900 with 1 Axes>

In [ ]:
 #| hide
import nbdev; nbdev.nbdev_export()